# Демо 3. SFT: від повного донавчання до LoRA і QLoRA

У цьому ноутбуці донавчаємо моделі **Qwen3** трьох розмірів виконувати одну вузьку прикладну задачу:
**класифікувати звернення клієнтів служби підтримки** й відповідати лише JSON із категорією та наміром звернення.

### Навіщо потрібен SFT

Інтернет-магазин щодня отримує тисячі повідомлень від клієнтів: скасувати замовлення, змінити адресу доставки,
повернути кошти тощо. Щоб автоматично передавати кожне звернення у відповідний відділ, програмі потрібна відповідь
у строго заданому форматі, наприклад `{"category": "ORDER", "intent": "cancel_order"}`, з мітками з переліку компанії.

Загальна instruct-модель може виконати таку задачу, якщо описати її в промпті, але:
- вона не знає переліку міток компанії, тому його доводиться передавати в кожному запиті;
- навіть із переліком вона часто плутає близькі мітки, бо не знає, як саме компанія їх розрізняє
  (наприклад, `get_refund` і `track_refund`).

Донавчання на прикладах із правильними мітками закріплює формат відповіді й значення кожної мітки у вагах моделі,
тому після нього достатньо короткого промпту.

План:
1. Підготувати дані у форматі `messages` і розбити їх на тренувальний, валідаційний і тестовий набори.
2. Підготувати функції для генерації й оцінювання.
3. Оцінити, чому повне донавчання великої моделі не вміщується в GPU T4.
4. Налаштувати **LoRA** і параметри тренування.
5. Донавчити методом LoRA моделі **Qwen3-0.6B, Qwen3-1.7B і Qwen3-4B** і порівняти, чи може менша модель після донавчання наздогнати більшу.
6. Виконати **повне донавчання** найменшої моделі й порівняти його з LoRA.
7. Зберегти адаптер, завантажити його знову й влити у ваги моделі.
8. Донавчити Qwen3-4B методом **QLoRA** і порівняти витрати памʼяті.

Приклад за мотивами розділу 11.4 [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/4)
і вправ курсу [smol-course](https://huggingface.co/learn/smol-course).
Для запуску в Colab оберіть середовище з GPU.

**Некрасов Артур Дмитрович**

Збережені CPU-вимірювання стосуються Qwen3-0.6B. У Colab із T4 виконуються також секції з 1.7B і 4B. Адаптери в секції 7 потрібні для повторного завантаження в межах ноутбука.

In [1]:
%%time
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("pip", 'install -q "transformers==5.20.2" "trl==1.17.1" "peft==0.21.2" "bitsandbytes==0.50.2" "datasets==5.1.0" "accelerate==1.15.0" psutil')
    get_ipython().run_line_magic("pip", 'uninstall -q -y torchao')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
CPU times: user 2.9 s, sys: 668 ms, total: 3.56 s
Wall time: 1min 2s


In [2]:
%%time
import copy
import gc
import json
import os
import threading
import time
from pathlib import Path

import pandas as pd
import psutil
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from trl import SFTConfig, SFTTrainer

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CPU_PROFILE = DEVICE == "cpu"
DTYPE = torch.float32 if CPU_PROFILE else torch.float16
if CPU_PROFILE:
    torch.set_num_threads(4)
    print("CPU: Qwen3-0.6B, 8 оновлень. LoRA, повне донавчання та NF4 QLoRA.")
else:
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Пам’ять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

GPU: Tesla T4
Пам’ять GPU: 15.64 GB
CPU times: user 28.8 s, sys: 2.88 s, total: 31.7 s
Wall time: 1min 1s


## 1. Дані

[Bitext Customer Support](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) —
датасет із майже 27 тис. повідомлень клієнтів англійською мовою. Кожне повідомлення має дві мітки:
- `category` — відділ, якого стосується звернення (11 категорій, наприклад `ORDER`, `REFUND`, `ACCOUNT`);
- `intent` — конкретний намір клієнта (27 намірів, наприклад `cancel_order`, `track_refund`).

Конкретні дані в повідомленнях замінено заповнювачами на кшталт `{{Order Number}}`.

In [3]:
%%time
raw_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
print(raw_dataset)

for ex in raw_dataset.shuffle(seed=0).select(range(5)):
    print(f"{ex['category']:<10} {ex['intent']:<25} {ex['instruction']}")

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})
SHIPPING   set_up_shipping_address   there are errors setting up a delivery address
PAYMENT    check_payment_methods     i want assistance seeing ur accepted payment opotions
SUBSCRIPTION newsletter_subscription   I have got to cancel the newsletter subscription
ACCOUNT    switch_account            can ya help me to change to the gold account
REFUND     track_refund              I am waiting for a compensation of {{Currency Symbol}}{{Refund Amount}}
CPU times: user 1.29 s, sys: 191 ms, total: 1.49 s
Wall time: 5.73 s


In [4]:
%%time
categories = sorted(set(raw_dataset["category"]))
intents = sorted(set(raw_dataset["intent"]))
print(f"Категорій: {len(categories)}, намірів: {len(intents)}")
print(categories)

Категорій: 11, намірів: 27
['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']
CPU times: user 962 ms, sys: 0 ns, total: 962 ms
Wall time: 1.2 s


### Дедуплікація

Частина звернень у датасеті повторюється дослівно. Якщо такий дублікат потрапить і в тренувальний, і у валідаційний набір,
validation loss буде заниженим: модель перевірятимуть на прикладах, які вона вже бачила.
Тому перед розбиттям залишаємо по одному прикладу для кожного тексту звернення (без урахування регістру й пробілів на краях).

In [5]:
%%time
seen, keep = set(), []
for i, text in enumerate(raw_dataset["instruction"]):
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        keep.append(i)

dedup_dataset = raw_dataset.select(keep)
print(f"До дедуплікації: {len(raw_dataset)}, після: {len(dedup_dataset)}")

До дедуплікації: 26872, після: 24371
CPU times: user 590 ms, sys: 2.06 ms, total: 592 ms
Wall time: 687 ms


### Формат `messages`

Кожне звернення перетворюємо на діалог із трьох повідомлень:
- `system` — коротка інструкція;
- `user` — повідомлення клієнта;
- `assistant` — правильна відповідь: JSON із мітками.

Колонка `chat_template_kwargs` вимикає режим міркувань Qwen3: модель має одразу відповідати JSON.

In [6]:
%%time
SHORT_PROMPT = "Classify the customer message. Reply with JSON only."


def to_messages(example):
    answer = json.dumps({"category": example["category"], "intent": example["intent"]})
    return {
        "messages": [
            {"role": "system", "content": SHORT_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": answer},
        ],
        "chat_template_kwargs": {"enable_thinking": False},
    }

CPU times: user 8 µs, sys: 0 ns, total: 8 µs
Wall time: 11 µs


### Тренувальний, валідаційний і тестовий набори

У профілі T4 після дедуплікації використовуються 2800 тренувальних, 200 валідаційних і 1000 тестових прикладів. У CPU-профілі використовуються 96, 16 і 16 прикладів відповідно. Межі фіксовані після перемішування з seed 42. Навчання використовує train, вибір і діагностика — validation. Тест використовується для порівняння готових варіантів. У CPU-профілі один приклад змінює частку на 6,25 відсоткового пункту.

In [7]:
%%time
shuffled = dedup_dataset.shuffle(seed=42)
train_size, validation_size, test_size = (96, 16, 16) if CPU_PROFILE else (2800, 200, 1000)
end_validation = train_size + validation_size
end_test = end_validation + test_size
train_raw = shuffled.select(range(train_size))
validation_raw = shuffled.select(range(train_size, end_validation))
test_raw = shuffled.select(range(end_validation, end_test))
split_keys = [set(text.strip().lower() for text in part["instruction"]) for part in [train_raw, validation_raw, test_raw]]
assert all(not split_keys[i] & split_keys[j] for i in range(3) for j in range(i + 1, 3))
train_dataset = train_raw.map(to_messages, remove_columns=shuffled.column_names)
val_dataset = validation_raw.map(to_messages, remove_columns=shuffled.column_names)
test_dataset = test_raw.map(to_messages, remove_columns=shuffled.column_names)
print(f"Тренувальний: {len(train_dataset)}, валідаційний: {len(val_dataset)}, тестовий: {len(test_dataset)}")
display(train_dataset[0]["messages"])

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Тренувальний: 2800, валідаційний: 200, тестовий: 1000


[{'role': 'system',
  'content': 'Classify the customer message. Reply with JSON only.'},
 {'role': 'user', 'content': "I'd like to donload the invoice #00108"},
 {'role': 'assistant',
  'content': '{"category": "INVOICE", "intent": "get_invoice"}'}]

CPU times: user 1.12 s, sys: 19.9 ms, total: 1.14 s
Wall time: 1.61 s


## 2. Функції для генерації й оцінювання

Порівнюємо три моделі однієї сімʼї з однаковою архітектурою й даними попереднього тренування — **Qwen3** на 0,6, 1,7 і 4 млрд параметрів.
Усі вони мають той самий токенізатор і шаблон розмови, тому токенізатор завантажуємо один раз.

In [8]:
%%time
MODELS = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]

tokenizer = AutoTokenizer.from_pretrained(MODELS[0])

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

CPU times: user 1.35 s, sys: 165 ms, total: 1.52 s
Wall time: 3.58 s


### Детальний промпт

Без донавчання модель не знає переліку міток, тому для оцінки «до» передаємо його в промпті разом з описом формату.

In [9]:
%%time
LONG_PROMPT = f"""Classify the customer message for the support ticket routing system.
Reply with a single JSON object and nothing else, in the format:
{{"category": "<CATEGORY>", "intent": "<intent>"}}

Allowed categories: {", ".join(categories)}
Allowed intents: {", ".join(intents)}"""

for name, prompt in [("Короткий промпт", SHORT_PROMPT), ("Детальний промпт", LONG_PROMPT)]:
    print(f"{name}: {len(tokenizer(prompt)['input_ids'])} токенів")

Короткий промпт: 11 токенів
Детальний промпт: 166 токенів
CPU times: user 3.14 ms, sys: 651 µs, total: 3.79 ms
Wall time: 8.92 ms


### Метрики

Відповідь моделі вважаємо правильною, якщо програма може розібрати її як JSON і мітки збігаються з еталонними.
До донавчання моделі Qwen3 часто загортають JSON у блок розмітки Markdown (` ```json … ``` `), тому таку обгортку перед розбором знімаємо,
але окремо рахуємо, як часто відповідь є чистим JSON без зайвого тексту.

Рахуємо чотири метрики на всьому тестовому наборі:
- частку відповідей, які є чистим JSON;
- частку відповідей, які після зняття обгортки є валідним JSON з полями `category` і `intent`;
- точність категорії;
- точність наміру.

Щоб оцінка 1000 прикладів не тривала надто довго, відповіді генеруються батчами по 32 звернення.

In [10]:
%%time
MAX_NEW_TOKENS = 48 if CPU_PROFILE else 60

def classify_batch(model, messages_batch, system_prompt, max_new_tokens=MAX_NEW_TOKENS):
    """Повертає відповіді моделі на список звернень клієнтів, згенеровані одним батчем."""
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        for message in messages_batch
    ]
    # Для генерації батчем короткі промпти доповнюються зліва, щоб відповіді всіх промптів починалися в одній позиції
    tokenizer.padding_side = "left"
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    tokenizer.padding_side = "right"
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id
        )
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    return [answer.strip() for answer in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]


def classify(model, message, system_prompt):
    """Повертає відповідь моделі на одне звернення клієнта."""
    return classify_batch(model, [message], system_prompt)[0]


def is_clean_json(answer):
    """Чи є відповідь JSON без жодного зайвого тексту."""
    try:
        json.loads(answer)
        return True
    except json.JSONDecodeError:
        return False


def parse(answer):
    """Знімає обгортку ```json … ```, розбирає JSON; повертає None, якщо формат неправильний."""
    text = answer.strip()
    if text.startswith("```"):
        text = text.strip("`").removeprefix("json").strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        return None
    if isinstance(result, dict) and {"category", "intent"} <= result.keys():
        return result
    return None


def evaluate(model, system_prompt, batch_size=8 if CPU_PROFILE else 32):
    """Рахує метрики на всьому тестовому наборі."""
    model.eval()
    messages = [ex["messages"][1]["content"] for ex in test_dataset]
    targets = [json.loads(ex["messages"][2]["content"]) for ex in test_dataset]
    answers = []
    for start in range(0, len(messages), batch_size):
        answers += classify_batch(model, messages[start:start + batch_size], system_prompt)

    clean = valid = category_ok = intent_ok = 0
    for answer, target in zip(answers, targets):
        clean += is_clean_json(answer)
        parsed = parse(answer)
        if parsed is not None:
            valid += 1
            category_ok += parsed["category"] == target["category"]
            intent_ok += parsed["intent"] == target["intent"]
    n = len(answers)
    metrics = {
        "Чистий JSON, %": 100 * clean / n,
        "Валідний JSON, %": 100 * valid / n,
        "Точність category, %": 100 * category_ok / n,
        "Точність intent, %": 100 * intent_ok / n,
    }
    return metrics, list(zip(messages, answers))


def show(answers, k=5):
    """Друкує перші k пар «звернення — відповідь моделі»."""
    for message, answer in answers[:k]:
        print(f"📩 {message}\n🤖 {answer}\n" + "-" * 60)


def free_memory():
    """Звільняє памʼять GPU після видалення моделей."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

CPU times: user 20 µs, sys: 0 ns, total: 20 µs
Wall time: 25.3 µs


## 3. Чому не повне донавчання

Під час повного донавчання з оптимізатором Adam у змішаній точності на кожен параметр потрібно приблизно 16 байт:

| Що зберігається | Байтів на параметр |
|---|---|
| ваги у fp16 | 2 |
| градієнти у fp16 | 2 |
| копія ваг у fp32: у fp16 малі оновлення ваг губляться через округлення | 4 |
| два стани Adam у fp32: усереднені градієнт і його квадрат, за якими Adam підбирає крок для кожного параметра | 8 |
| **разом** | **16** |

До цього додаються активації, які залежать від довжини послідовностей і розміру батча.

Для Qwen3-4B це близько 64 GB без активацій, тоді як T4 має 15 GB: повне донавчання такої моделі потребує GPU рівня A100 на 80 GB.
Для Qwen3-0.6B потрібно лише близько 10 GB, тому в розділі 6 виконаємо повне донавчання саме цієї моделі.

## 4. Налаштування LoRA і тренування

LoRA заморожує всі ваги моделі й додає до вибраних шарів дві невеликі матриці $A$ і $B$.
Тренуються лише вони, тому градієнти й стани оптимізатора потрібні тільки для них.

Параметри `LoraConfig`:
- `r` — ранг матриць адаптера: що більший, то більше параметрів і «свободи» має адаптер;
- `lora_alpha` — масштаб внеску адаптера (зазвичай `2 × r`);
- `lora_dropout` — dropout в адаптері для регуляризації;
- `target_modules="all-linear"` — адаптери додаються до всіх лінійних шарів (attention і MLP).

In [11]:
%%time
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

CPU times: user 102 µs, sys: 10 µs, total: 112 µs
Wall time: 116 µs


Параметри `SFTConfig` для T4: 100 оновлень, батч 4, накопичення градієнта 2, перевірка кожні 10 кроків, fp16 та gradient checkpointing. CPU-профіль використовує 8 оновлень, батч 2, накопичення 2, перевірку кожні 4 кроки і fp32. У LoRA та QLoRA використано однакові r=16, alpha=32 і learning rate 2e-4. Обидва профілі застосовують `assistant_only_loss=True`. Нижче перевіряється, що в батчі є як токени з цільовою міткою, так і токени, замасковані значенням −100.

In [12]:
%%time
def make_config(output_dir, learning_rate, full):
    return SFTConfig(
        output_dir=output_dir,
        max_length=128 if CPU_PROFILE else 256,
        per_device_train_batch_size=2 if CPU_PROFILE else 4,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=4 if CPU_PROFILE else 8,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=1 if CPU_PROFILE else 5,
        max_steps=8 if CPU_PROFILE else 100,
        eval_strategy="steps",
        eval_steps=4 if CPU_PROFILE else 10,
        fp16=not CPU_PROFILE,
        bf16=False,
        use_cpu=CPU_PROFILE,
        gradient_checkpointing=not CPU_PROFILE,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        assistant_only_loss=True,
        logging_steps=2 if CPU_PROFILE else 10,
        save_strategy="no",
        report_to="none",
        disable_tqdm=True,
        dataloader_pin_memory=not CPU_PROFILE,
        optim="sgd" if CPU_PROFILE and full else "adamw_torch",
        seed=42,
        data_seed=42,
    )

def train(model, output_dir, adapter_config=peft_config, learning_rate=2e-4):
    if not CPU_PROFILE:
        torch.cuda.reset_peak_memory_stats()
    trainer = SFTTrainer(model=model, args=make_config(output_dir, learning_rate, adapter_config is None),
        train_dataset=train_dataset, eval_dataset=val_dataset, processing_class=tokenizer,
        peft_config=copy.deepcopy(adapter_config))
    if getattr(model, "is_loaded_in_4bit", False):
        for parameter in trainer.model.parameters():
            if parameter.requires_grad:
                parameter.data = parameter.data.float()
    if adapter_config is not None:
        trainer.model.print_trainable_parameters()
    labels = trainer.data_collator([trainer.train_dataset[0]])["labels"]
    target_tokens = int((labels != -100).sum())
    assert 0 < target_tokens < labels.numel()
    print(f"Цільові токени в контрольному рядку: {target_tokens} з {labels.numel()}")
    process = psutil.Process()
    rss = [process.memory_info().rss]
    finished = threading.Event()

    def sample_memory():
        while not finished.wait(0.1):
            rss[0] = max(rss[0], process.memory_info().rss)

    monitor = threading.Thread(target=sample_memory, daemon=True)
    monitor.start()
    if not CPU_PROFILE:
        torch.cuda.synchronize()
    started = time.perf_counter()
    try:
        trainer.train()
    finally:
        if not CPU_PROFILE:
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        finished.set()
        monitor.join()
    eval_losses = [entry["eval_loss"] for entry in trainer.state.log_history if "eval_loss" in entry]
    stats = {"Тривалість, хв": elapsed / 60,
        "Пікова пам’ять GPU, GB": None if CPU_PROFILE else torch.cuda.max_memory_allocated() / 1e9,
        "Пік RSS процесу, GB": rss[0] / 1e9,
        "Validation loss": eval_losses[-1], "Токенів із loss у контрольному рядку": target_tokens}
    return trainer, stats

load_records = []

def record_load(model, label):
    values = {"варіант": label, "розміщені параметри, GB": sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9,
        "RSS після завантаження, GB": psutil.Process().memory_info().rss / 1e9,
        "VRAM після завантаження, GB": None if CPU_PROFILE else torch.cuda.memory_allocated() / 1e9}
    load_records.append(values)
    print(f"{label}. Розміщені параметри: {values['розміщені параметри, GB']:.3f} GB.")
    if CPU_PROFILE:
        print(f"RSS після завантаження: {values['RSS після завантаження, GB']:.3f} GB.")
    else:
        print(f"Пам’ять GPU після завантаження: {values['VRAM після завантаження, GB']:.3f} GB.")

CPU times: user 10 µs, sys: 0 ns, total: 10 µs
Wall time: 13.1 µs


## 5. LoRA на моделях різного розміру

T4-профіль запускає Qwen3-0.6B, 1.7B і 4B у fp16. CPU-профіль запускає Qwen3-0.6B у fp32. Порівняння лекції оцінює базову модель з детальним промптом і донавчену з коротким. У CPU-профілі також вимірюється початкова модель з коротким промптом. Адаптери зберігаються в робочому середовищі для секції 7.

In [13]:
%%time
results = {}
loss_histories = {}

def run_lora(model_name):
    set_seed(42)
    short_name = model_name.split("/")[-1]
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=DTYPE, device_map=DEVICE)
    record_load(model, short_name + " LoRA base")
    before_metrics, before_answers = evaluate(model, LONG_PROMPT)
    print("До донавчання, детальний промпт")
    show(before_answers, k=3)
    comparison = {"до, детальний": before_metrics}
    if CPU_PROFILE:
        before_short, _ = evaluate(model, SHORT_PROMPT)
        comparison["до, короткий"] = before_short

    trainer, stats = train(model, f"{short_name}-lora")
    loss_histories[short_name + " LoRA"] = list(trainer.state.log_history)
    after_metrics, after_answers = evaluate(trainer.model, SHORT_PROMPT)
    print("Після донавчання, короткий промпт")
    show(after_answers, k=3)
    adapter_dir = f"{short_name}-lora-adapter"
    trainer.save_model(adapter_dir)
    stats["Адаптер, MB"] = Path(adapter_dir, "adapter_model.safetensors").stat().st_size / 1e6
    comparison["LoRA"] = after_metrics | stats
    results[short_name] = comparison
    display(pd.DataFrame(comparison).round(3))
    del trainer, model
    free_memory()

CPU times: user 8 µs, sys: 0 ns, total: 8 µs
Wall time: 11 µs


In [14]:
%%time
run_lora("Qwen/Qwen3-0.6B")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-0.6B LoRA base. Розміщені параметри: 1.192 GB.
Пам’ять GPU після завантаження: 1.193 GB.
До донавчання, детальний промпт
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------


Tokenizing train dataset:   0%|          | 0/2800 [00:00<?, ? examples/s]

... (6 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650
Цільові токени в контрольному рядку: 20 з 58
{'loss': '1.907', 'grad_norm': '2.579', 'learning_rate': '0.0001995', 'entropy': '0.6402', 'mean_token_accuracy': '0.7175', 'num_tokens': '4329', 'epoch': '0.02857'}
{'eval_loss': '0.3652', 'eval_runtime': '3.34', 'eval_samples_per_second': '59.89', 'eval_steps_per_second': '7.486', 'eval_entropy': '0.3907', 'eval_mean_token_accuracy': '0.9183', 'eval_num_tokens': '4329', 'epoch': '0.02857'}
{'loss': '0.2522', 'grad_norm': '1.714', 'learning_rate': '0.0001909', 'entropy': '0.2888', 'mean_token_accuracy': '0.9363', 'num_tokens': '8680', 'epoch': '0.05714'}
{'eval_loss': '0.1043', 'eval_runtime': '2.959', 'eval_samples_per_second': '67.59', 'eval_steps_per_second': '8.449', 'eval_entropy': '0.1329', 'eval_mean_token_accuracy': '0.9715', 'eval_num_tokens': '8680', 'epoch': '0.05714'}
{'loss': '0.07856', 'grad_norm': '1.654', 'learning_rate': '0.0001724', 'entropy': '0

,"до, детальний",LoRA
"Чистий JSON, %",100.0,100.000
"Валідний JSON, %",100.0,100.000
"Точність category, %",37.9,99.700
"Точність intent, %",25.6,97.500
"Тривалість, хв",NaN,2.936
"Пікова пам’ять GPU, GB",NaN,1.423
"Пік RSS процесу, GB",NaN,2.513
Validation loss,NaN,0.008
Токенів із loss у контрольному рядку,NaN,20.000
"Адаптер, MB",NaN,40.422


CPU times: user 4min 28s, sys: 8.87 s, total: 4min 36s
Wall time: 5min 26s


In [15]:
%%time
if CPU_PROFILE:
    print("Qwen3-1.7B: клітинка для T4-профілю.")
else:
    run_lora("Qwen/Qwen3-1.7B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-1.7B LoRA base. Розміщені параметри: 3.441 GB.
Пам’ять GPU після завантаження: 3.460 GB.
До донавчання, детальний промпт
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "contact_customer_service"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------
trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030
Цільові токени в контрольному рядку: 20 з 58
{'loss': '2.148', 'grad_norm': '1.194', 'learning_rate': '0.0001991', 'entropy': '0.3524', 'mean_token_accuracy': '0.7712', 'num_tokens': '4329', 'epoch': '0.02857'}
{'eval_loss': '0.3093', 'eval_runtime': '4.018', 'eval_samples_per_second

,"до, детальний",LoRA
"Чистий JSON, %",100.0,100.000
"Валідний JSON, %",100.0,100.000
"Точність category, %",62.7,99.600
"Точність intent, %",57.4,97.900
"Тривалість, хв",NaN,2.565
"Пікова пам’ять GPU, GB",NaN,3.831
"Пік RSS процесу, GB",NaN,2.581
Validation loss,NaN,0.007
Токенів із loss у контрольному рядку,NaN,20.000
"Адаптер, MB",NaN,69.782


CPU times: user 5min 5s, sys: 25 s, total: 5min 30s
Wall time: 6min 32s


In [16]:
%%time
if CPU_PROFILE:
    print("Qwen3-4B: клітинка для T4-профілю.")
else:
    run_lora("Qwen/Qwen3-4B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-4B LoRA base. Розміщені параметри: 8.045 GB.
Пам’ять GPU після завантаження: 8.062 GB.
До донавчання, детальний промпт
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
Цільові токени в контрольному рядку: 20 з 58
{'loss': '1.978', 'grad_norm': '1.271', 'learning_rate': '0.0001991', 'entropy': '0.3804', 'mean_token_accuracy': '0.7655', 'num_tokens': '4329', 'epoch': '0.02857'}
{'eval_loss': '0.2138', 'eval_runtime': '8.58', 'eval_samples_per_second': '23.31', 'eval_ste

,"до, детальний",LoRA
"Чистий JSON, %",100.0,100.000
"Валідний JSON, %",100.0,100.000
"Точність category, %",88.1,99.700
"Точність intent, %",72.5,97.600
"Тривалість, хв",NaN,3.903
"Пікова пам’ять GPU, GB",NaN,8.746
"Пік RSS процесу, GB",NaN,2.563
Validation loss,NaN,0.005
Токенів із loss у контрольному рядку,NaN,20.000
"Адаптер, MB",NaN,132.188


CPU times: user 8min 13s, sys: 1min 22s, total: 9min 35s
Wall time: 13min 31s


### Порівняння

Чи може менша модель після донавчання досягти якості більшої? Порівняйте точність, тривалість тренування, памʼять і розмір адаптера.

In [17]:
%%time
display(pd.DataFrame({f"{name} {stage}": metrics for name, stages in results.items()
    for stage, metrics in stages.items()}).round(3))

,"Qwen3-0.6B до, детальний",Qwen3-0.6B LoRA,"Qwen3-1.7B до, детальний",Qwen3-1.7B LoRA,"Qwen3-4B до, детальний",Qwen3-4B LoRA
"Чистий JSON, %",100.0,100.000,100.0,100.000,100.0,100.000
"Валідний JSON, %",100.0,100.000,100.0,100.000,100.0,100.000
"Точність category, %",37.9,99.700,62.7,99.600,88.1,99.700
"Точність intent, %",25.6,97.500,57.4,97.900,72.5,97.600
"Тривалість, хв",NaN,2.936,NaN,2.565,NaN,3.903
"Пікова пам’ять GPU, GB",NaN,1.423,NaN,3.831,NaN,8.746
"Пік RSS процесу, GB",NaN,2.513,NaN,2.581,NaN,2.563
Validation loss,NaN,0.008,NaN,0.007,NaN,0.005
Токенів із loss у контрольному рядку,NaN,20.000,NaN,20.000,NaN,20.000
"Адаптер, MB",NaN,40.422,NaN,69.782,NaN,132.188


CPU times: user 18.5 ms, sys: 834 µs, total: 19.3 ms
Wall time: 20.4 ms


## 6. Повне донавчання Qwen3-0.6B

Навчаються всі параметри. Ваги завантажуються у fp32, learning rate становить 2e-5. Для T4 застосовується AdamW зі змішаною точністю. Для CPU використано SGD без додаткових моментів оптимізатора, щоб вмістити ваги й градієнти у RAM. Різницю оптимізаторів слід врахувати при порівнянні повного донавчання з LoRA.

In [18]:
%%time
set_seed(42)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float32, device_map=DEVICE)
record_load(model, "Qwen3-0.6B повне")
trainer, full_stats = train(model, "Qwen3-0.6B-full", adapter_config=None, learning_rate=2e-5)
loss_histories["Qwen3-0.6B повне"] = list(trainer.state.log_history)
full_metrics, full_answers = evaluate(trainer.model, SHORT_PROMPT)
show(full_answers, k=3)
display(pd.DataFrame({"Qwen3-0.6B LoRA": results["Qwen3-0.6B"]["LoRA"],
    "Qwen3-0.6B повне": full_metrics | full_stats}).round(3))
del trainer, model
free_memory()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3-0.6B повне. Розміщені параметри: 2.384 GB.
Пам’ять GPU після завантаження: 2.403 GB.
Цільові токени в контрольному рядку: 20 з 58
{'loss': '2.213', 'grad_norm': '24.33', 'learning_rate': '1.999e-05', 'entropy': '0.6291', 'mean_token_accuracy': '0.6798', 'num_tokens': '4329', 'epoch': '0.02857'}
{'eval_loss': '0.2975', 'eval_runtime': '1.991', 'eval_samples_per_second': '100.5', 'eval_steps_per_second': '12.56', 'eval_entropy': '0.3823', 'eval_mean_token_accuracy': '0.933', 'eval_num_tokens': '4329', 'epoch': '0.02857'}
{'loss': '0.2071', 'grad_norm': '10.79', 'learning_rate': '1.935e-05', 'entropy': '0.2195', 'mean_token_accuracy': '0.9494', 'num_tokens': '8680', 'epoch': '0.05714'}
{'eval_loss': '0.07074', 'eval_runtime': '2.014', 'eval_samples_per_second': '99.31', 'eval_steps_per_second': '12.41', 'eval_entropy': '0.07222', 'eval_mean_token_accuracy': '0.9744', 'eval_num_tokens': '8680', 'epoch': '0.05714'}
{'loss': '0.0625', 'grad_norm': '5.938', 'learning_rate': '1.768e-05',

,Qwen3-0.6B LoRA,Qwen3-0.6B повне
"Чистий JSON, %",100.000,100.000
"Валідний JSON, %",100.000,100.000
"Точність category, %",99.700,99.200
"Точність intent, %",97.500,98.000
"Тривалість, хв",2.936,1.847
"Пікова пам’ять GPU, GB",1.423,11.941
"Пік RSS процесу, GB",2.513,2.618
Validation loss,0.008,0.004
Токенів із loss у контрольному рядку,20.000,20.000
"Адаптер, MB",40.422,NaN


CPU times: user 2min 34s, sys: 2.19 s, total: 2min 37s
Wall time: 2min 41s


## 7. Збереження і завантаження адаптера

`save_model` зберігає **лише адаптер** (`adapter_model.safetensors`) разом із його налаштуваннями й токенізатором, а не всю модель.

In [19]:
%%time
comparison_model = "Qwen/Qwen3-0.6B" if CPU_PROFILE else "Qwen/Qwen3-4B"
comparison_short_name = comparison_model.split("/")[-1]
adapter_dir = Path(f"{comparison_short_name}-lora-adapter")
for file in sorted(adapter_dir.iterdir()):
    print(f"{file.name:<35} {file.stat().st_size / 1e6:>8.2f} MB")

README.md                               0.01 MB
adapter_config.json                     0.00 MB
adapter_model.safetensors             132.19 MB
chat_template.jinja                     0.00 MB
tokenizer.json                         11.42 MB
tokenizer_config.json                   0.00 MB
training_args.bin                       0.01 MB
CPU times: user 1.24 ms, sys: 0 ns, total: 1.24 ms
Wall time: 1.32 ms


Щоб використати адаптер, завантажуємо модель і накладаємо на неї адаптер з диска.
Метод `merge_and_unload` вливає адаптер у ваги моделі: $W' = W + \frac{\alpha}{r} BA$.
Після цього модель працює як звичайна, без додаткових обчислень для адаптера.

In [20]:
%%time
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained(comparison_model, dtype=DTYPE, device_map=DEVICE)
model = PeftModel.from_pretrained(model, adapter_dir)
model = model.merge_and_unload()
model.eval()
print(type(model).__name__)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM
CPU times: user 9.41 s, sys: 5.87 s, total: 15.3 s
Wall time: 34.3 s


Перевіримо злиту модель на власних повідомленнях:

In [21]:
%%time
my_messages = [
    "Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?",
    "I was charged twice for the same order, please help",
    "How can I talk to a real person?",
    "I guess, you have shipped to a wrong address",
]
for message in my_messages:
    print(f"📩 {message}\n🤖 {classify(model, message, SHORT_PROMPT)}\n")

📩 Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?
🤖 {"category": "DELIVERY", "intent": "delivery_period"}

📩 I was charged twice for the same order, please help
🤖 {"category": "ORDER", "intent": "change_order"}

📩 How can I talk to a real person?
🤖 {"category": "CONTACT", "intent": "contact_human_agent"}

📩 I guess, you have shipped to a wrong address
🤖 {"category": "DELIVERY", "intent": "delivery_issue"}

CPU times: user 3.7 s, sys: 5.41 ms, total: 3.7 s
Wall time: 3.74 s


In [22]:
%%time
del model
free_memory()

CPU times: user 416 ms, sys: 141 ms, total: 558 ms
Wall time: 557 ms


## 8. QLoRA

QLoRA зберігає заморожені лінійні ваги в NF4. Для блоку ваг зберігається масштаб, а 4-бітний індекс вказує на одне з 16 значень коду. Розміщення значень NF4 розраховане на ваги з приблизно нормальним розподілом. Double quantization стискає також масштаби. У матричних операціях використовується обчислювальний dtype, а LoRA-адаптери оновлюються з градієнтами. На T4 порівнюється Qwen3-4B, на CPU — Qwen3-0.6B. [QLoRA](https://arxiv.org/abs/2305.14314) · [Документація bitsandbytes](https://huggingface.co/docs/transformers/main/en/quantization/bitsandbytes)

In [23]:
%%time
import bitsandbytes as bnb

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
set_seed(42)
model = AutoModelForCausalLM.from_pretrained(comparison_model, dtype=DTYPE,
    quantization_config=bnb_config, device_map=DEVICE)
quantized_layers = sum(isinstance(layer, bnb.nn.Linear4bit) for layer in model.modules())
assert quantized_layers > 0
print("Шарів Linear4bit:", quantized_layers)
record_load(model, comparison_short_name + " QLoRA base")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Шарів Linear4bit: 252
Qwen3-4B QLoRA base. Розміщені параметри: 2.595 GB.
Пам’ять GPU після завантаження: 2.697 GB.
CPU times: user 8.82 s, sys: 5.44 s, total: 14.3 s
Wall time: 33.6 s


In [24]:
%%time
trainer, qlora_stats = train(model, f"{comparison_short_name}-qlora")
loss_histories[comparison_short_name + " QLoRA"] = list(trainer.state.log_history)
qlora_metrics, qlora_answers = evaluate(trainer.model, SHORT_PROMPT)
show(qlora_answers, k=3)
display(pd.DataFrame([qlora_stats | qlora_metrics]).round(3))
del trainer, model
free_memory()

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
Цільові токени в контрольному рядку: 20 з 58
{'loss': '1.988', 'grad_norm': '1.152', 'learning_rate': '0.0001991', 'entropy': '0.4694', 'mean_token_accuracy': '0.75', 'num_tokens': '4329', 'epoch': '0.02857'}
{'eval_loss': '0.2635', 'eval_runtime': '9.29', 'eval_samples_per_second': '21.53', 'eval_steps_per_second': '2.691', 'eval_entropy': '0.2975', 'eval_mean_token_accuracy': '0.9349', 'eval_num_tokens': '4329', 'epoch': '0.02857'}
{'loss': '0.174', 'grad_norm': '0.6683', 'learning_rate': '0.0001895', 'entropy': '0.1838', 'mean_token_accuracy': '0.9518', 'num_tokens': '8680', 'epoch': '0.05714'}
{'eval_loss': '0.07052', 'eval_runtime': '9.697', 'eval_samples_per_second': '20.62', 'eval_steps_per_second': '2.578', 'eval_entropy': '0.09436', 'eval_mean_token_accuracy': '0.9781', 'eval_num_tokens': '8680', 'epoch': '0.05714'}
{'loss': '0.05067', 'grad_norm': '1.427', 'learning_rate': '0.0001701', 'entropy': 

,"Тривалість, хв","Пікова пам’ять GPU, GB","Пік RSS процесу, GB",Validation loss,Токенів із loss у контрольному рядку,"Чистий JSON, %","Валідний JSON, %","Точність category, %","Точність intent, %"
0,4.706,3.407,2.709,0.004,20,100.0,100.0,99.5,97.8


CPU times: user 6min 19s, sys: 27.4 s, total: 6min 47s
Wall time: 6min 49s


## 9. Підсумкове порівняння

In [25]:
%%time
summary = {f"{name} {stage}": metrics for name, stages in results.items() for stage, metrics in stages.items()}
summary["Qwen3-0.6B повне"] = full_metrics | full_stats
summary[comparison_short_name + " QLoRA"] = qlora_metrics | qlora_stats
display(pd.DataFrame(summary).T.round(3))
display(pd.DataFrame(load_records).round(3))
for name, history in loss_histories.items():
    print(name)
    display(pd.DataFrame([row for row in history if "loss" in row or "eval_loss" in row]).round(4))

,"Чистий JSON, %","Валідний JSON, %","Точність category, %","Точність intent, %","Тривалість, хв","Пікова пам’ять GPU, GB","Пік RSS процесу, GB",Validation loss,Токенів із loss у контрольному рядку,"Адаптер, MB"
"Qwen3-0.6B до, детальний",100.0,100.0,37.9,25.6,NaN,NaN,NaN,NaN,NaN,NaN
Qwen3-0.6B LoRA,100.0,100.0,99.7,97.5,2.936,1.423,2.513,0.008,20.0,40.422
"Qwen3-1.7B до, детальний",100.0,100.0,62.7,57.4,NaN,NaN,NaN,NaN,NaN,NaN
Qwen3-1.7B LoRA,100.0,100.0,99.6,97.9,2.565,3.831,2.581,0.007,20.0,69.782
"Qwen3-4B до, детальний",100.0,100.0,88.1,72.5,NaN,NaN,NaN,NaN,NaN,NaN
Qwen3-4B LoRA,100.0,100.0,99.7,97.6,3.903,8.746,2.563,0.005,20.0,132.188
Qwen3-0.6B повне,100.0,100.0,99.2,98.0,1.847,11.941,2.618,0.004,20.0,NaN
Qwen3-4B QLoRA,100.0,100.0,99.5,97.8,4.706,3.407,2.709,0.004,20.0,NaN


,варіант,"розміщені параметри, GB","RSS після завантаження, GB","VRAM після завантаження, GB"
0,Qwen3-0.6B LoRA base,1.192,1.522,1.193
1,Qwen3-1.7B LoRA base,3.441,2.675,3.460
2,Qwen3-4B LoRA base,8.045,2.544,8.062
3,Qwen3-0.6B повне,2.384,2.611,2.403
4,Qwen3-4B QLoRA base,2.595,2.681,2.697


Qwen3-0.6B LoRA


,loss,grad_norm,learning_rate,entropy,mean_token_accuracy,num_tokens,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,eval_entropy,eval_mean_token_accuracy,eval_num_tokens
0,1.9074,2.5793,0.0002,0.6402,0.7175,4329.0,0.0286,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,0.0286,10,0.3652,3.3397,59.887,7.486,0.3907,0.9183,4329.0
2,0.2522,1.7141,0.0002,0.2888,0.9363,8680.0,0.0571,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,0.0571,20,0.1043,2.9590,67.590,8.449,0.1329,0.9715,8680.0
4,0.0786,1.6537,0.0002,0.0832,0.9796,13009.0,0.0857,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,0.0857,30,0.0863,2.7489,72.755,9.094,0.0545,0.9762,13009.0
6,0.0329,0.2860,0.0001,0.0458,0.9925,17374.0,0.1143,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,0.1143,40,0.0337,2.7445,72.874,9.109,0.0431,0.9906,17374.0
8,0.0273,0.1458,0.0001,0.0348,0.9938,21702.0,0.1429,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,0.1429,50,0.0241,2.6932,74.261,9.283,0.0347,0.9938,21702.0


Qwen3-1.7B LoRA


,loss,grad_norm,learning_rate,entropy,mean_token_accuracy,num_tokens,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,eval_entropy,eval_mean_token_accuracy,eval_num_tokens
0,2.1477,1.1936,0.0002,0.3524,0.7712,4329.0,0.0286,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,0.0286,10,0.3093,4.0177,49.780,6.223,0.3036,0.9282,4329.0
2,0.2089,0.8928,0.0002,0.2195,0.9451,8680.0,0.0571,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,0.0571,20,0.1030,3.8811,51.532,6.441,0.1462,0.9704,8680.0
4,0.0654,1.2708,0.0002,0.0888,0.9781,13009.0,0.0857,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,0.0857,30,0.0486,3.8382,52.107,6.513,0.0420,0.9853,13009.0
6,0.0221,0.5579,0.0001,0.0283,0.9931,17374.0,0.1143,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,0.1143,40,0.0283,3.9488,50.649,6.331,0.0214,0.9911,17374.0
8,0.0235,0.1698,0.0001,0.0223,0.9944,21702.0,0.1429,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,0.1429,50,0.0219,4.0460,49.431,6.179,0.0135,0.9955,21702.0


... (3 outputs left out)

,loss,grad_norm,learning_rate,entropy,mean_token_accuracy,num_tokens,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,eval_entropy,eval_mean_token_accuracy,eval_num_tokens
0,2.2134,24.3259,0.0,0.6291,0.6798,4329.0,0.0286,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,0.0286,10,0.2975,1.9910,100.451,12.556,0.3823,0.9330,4329.0
2,0.2071,10.7906,0.0,0.2195,0.9494,8680.0,0.0571,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,0.0571,20,0.0707,2.0138,99.315,12.414,0.0722,0.9744,8680.0
4,0.0625,5.9383,0.0,0.0446,0.9808,13009.0,0.0857,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,0.0857,30,0.0561,1.9896,100.524,12.565,0.0343,0.9866,13009.0
6,0.0279,1.4911,0.0,0.0318,0.9925,17374.0,0.1143,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,0.1143,40,0.0442,1.8583,107.627,13.453,0.0177,0.9908,17374.0
8,0.0261,2.0698,0.0,0.0178,0.9957,21702.0,0.1429,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,0.1429,50,0.0229,1.8333,109.095,13.637,0.0178,0.9948,21702.0


Qwen3-4B QLoRA


,loss,grad_norm,learning_rate,entropy,mean_token_accuracy,num_tokens,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,eval_entropy,eval_mean_token_accuracy,eval_num_tokens
0,1.9880,1.1517,0.0002,0.4694,0.7500,4329.0,0.0286,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,0.0286,10,0.2635,9.2902,21.528,2.691,0.2975,0.9349,4329.0
2,0.1740,0.6683,0.0002,0.1838,0.9518,8680.0,0.0571,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,0.0571,20,0.0705,9.6971,20.625,2.578,0.0944,0.9781,8680.0
4,0.0507,1.4267,0.0002,0.0530,0.9858,13009.0,0.0857,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,0.0857,30,0.0510,10.5559,18.947,2.368,0.0293,0.9868,13009.0
6,0.0186,0.1354,0.0001,0.0191,0.9932,17374.0,0.1143,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,0.1143,40,0.0372,10.4819,19.080,2.385,0.0226,0.9906,17374.0
8,0.0225,0.2040,0.0001,0.0221,0.9944,21702.0,0.1429,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,0.1429,50,0.0160,10.1043,19.794,2.474,0.0189,0.9953,21702.0


CPU times: user 230 ms, sys: 14.3 ms, total: 244 ms
Wall time: 245 ms


Порівняй LoRA й QLoRA на одному checkpoint, однакових наборах і короткому промпті. Пік пам’яті, час і точність залежать від пристрою, dtype, реалізації деквантизації та кількості прикладів. У CPU-профілі вимірюється RSS процесу. На T4 вимірюється виділена VRAM. Для перевірки впливу самого донавчання на CPU доступні також передбачення базової моделі з коротким промптом.

## 0.1. Відповіді на питання 22–30

### 22. Час виконання

Клітинки виводять `Wall time`, а таблиця тренувань містить час циклу навчання й валідації. Сума часу CPU-клітинок становить **368.13 с**. У цьому профілі для 0.6B виконано LoRA, повне донавчання, повторне завантаження адаптера й QLoRA. Використано 96 train, 16 validation та 16 test. Під час імпорту bitsandbytes повідомила про відсутність необов’язкового оптимізованого CPU-ядра. Квантизовані шари й навчання відпрацювали на доступному backend. Розміри 1.7B і 4B запускаються в профілі T4.

### 23. Пам’ять після завантаження

Орієнтир для ваг fp16 — кількість параметрів, помножена на два байти. Для 4B це близько 8 GB, для 1.7B близько 3,4 GB, для 0.6B близько 1,2 GB. На CPU fp32 потребує приблизно чотири байти на параметр. Поточний замір GPU показує виділені тензори процесу після завантаження. Пік під час тренування включає активації, градієнти, буфери й стан оптимізатора. Параметри в NF4 займають менше, але частина тензорів і метадані квантизації зберігаються з іншою точністю. RSS процесу охоплює також пам’ять Python, бібліотек і буферів. У фактичному CPU-запуску сума байтів розміщених параметрів 0.6B становить 2.384 GB у fp32 і 0.843 GB після завантаження NF4. Це узгоджується з різницею форматів. Квантизовано 196 шарів `Linear4bit`. VRAM-показник заповнюється при запуску на T4.

### 24. `torch.no_grad()`

Під час генерації цей контекст вимикає запис графа автоматичного диференціювання. Це зменшує пам’ять та витрати інференсу. `model.eval()` окремо переводить dropout та інші шари в режим оцінювання.

### 25. `do_sample=False`

Генерація вибирає найімовірніший токен без випадкової вибірки. Для класифікації це стабілізує порівняння формату й міток. На однаковій конфігурації greedy-декодування зменшує варіативність повторних запусків.

### 26. Чи підходить голова класифікації

Для фіксованих наборів `category` та `intent` можна навчити енкодер з однією спільною або двома окремими головами й сформувати JSON у коді. У цьому демо перевіряється інша постановка — генеративна модель зберігає формат чат-асистента й навчається повертати JSON як відповідь. SFT з LoRA дозволяє адаптувати цю поведінку з невеликим числом нових параметрів.

### 27. Короткий і детальний промпти

Порівняння робиться на тому самому тексті звернення й тому самому чат-шаблоні. Додаткові вхідні токени на день дорівнюють різниці довжин, помноженій на 10 000. Для тарифу за мільйон вхідних токенів множимо тариф на цей добовий приріст, поділений на мільйон. Локальний запуск використовує обчислення й пам’ять пропорційно довжині входу. Кешування префікса може змінити практичні витрати.

У виміряному прикладі короткий вхід займає 39 токенів, детальний — 194. Детальний промпт додає 1,550,000 токенів на 10 000 звернень.

In [26]:
%%time
example_message = test_dataset[0]["messages"][1]["content"]
prompt_counts = {}
for label, instruction in [("короткий", SHORT_PROMPT), ("детальний", LONG_PROMPT)]:
    messages = [{"role": "system", "content": instruction}, {"role": "user", "content": example_message}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    prompt_counts[label] = len(tokenizer.encode(formatted, add_special_tokens=False))
extra_tokens = prompt_counts["детальний"] - prompt_counts["короткий"]
display(pd.DataFrame([{"Промпт": label, "Токенів на звернення": count,
    "Токенів на 10 000 звернень": count * 10_000} for label, count in prompt_counts.items()]))
print(f"Різниця: {extra_tokens} токенів на звернення, {extra_tokens * 10_000:,} на день.")

sft_measurements = {"device": DEVICE, "train_size": train_size, "validation_size": validation_size,
    "test_size": test_size, "load_records": load_records, "summary": summary,
    "loss_histories": loss_histories, "prompt_counts": prompt_counts, "daily_difference": extra_tokens * 10_000,
    "quantized_layers": quantized_layers, "comparison_model": comparison_model}
Path("sft_demo_measurements.json").write_text(json.dumps(sft_measurements, ensure_ascii=False, indent=2), encoding="utf-8")

,Промпт,Токенів на звернення,Токенів на 10 000 звернень
0,короткий,37,370000
1,детальний,192,1920000


Різниця: 155 токенів на звернення, 1,550,000 на день.
CPU times: user 15.6 ms, sys: 158 µs, total: 15.8 ms
Wall time: 18.9 ms


39803

### 28. `enable_thinking=False`

Тренувальна відповідь містить лише JSON. `chat_template_kwargs` задає шаблон без режиму міркування для кожного рядка, щоб формат під час SFT збігався з інференсом. У режимі `/think` модель очікує префікс або блок міркувань. Якщо цільові дані залишаться тільки JSON, виникне невідповідність формату. Модель може навчитися пропускати або нестабільно завершувати цей блок. Для тренування міркувань потрібні окремі відповідно розмічені приклади.

### 29. `assistant_only_loss=True`

Мітки поза відповідями асистента маскуються значенням −100. Модель бачить системну інструкцію й звернення в контексті, а loss обчислюється на цільових токенах асистента. Без маски для такого conversational dataset loss зазвичай охоплює всю серіалізовану послідовність. Частина сигналу йшла б на відтворення тексту користувача й службових фрагментів, хоча ціль задачі — передбачення JSON. Для підтримуваних сімейств на кшталт Qwen3 TRL додає потрібні межі генерації в тренувальному шаблоні. [SFT Trainer](https://huggingface.co/docs/trl/en/sft_trainer)

### 30. LoRA і QLoRA

В обох випадках оновлюються адаптери. QLoRA стискає заморожені базові ваги у NF4, тому зменшує обсяг зберігання. Час залежить від ядер і деквантизації. Порівнювати якість слід на однаковій моделі, наборах і промпті. QLoRA доцільна, коли вага звичайної моделі разом з активаціями не вміщується у VRAM або коли потрібно використати більший контекст чи батч у тому самому бюджеті.

У CPU-порівнянні Qwen3-0.6B обидва адаптери тренувалися 8 кроків і тестувалися на 16 зверненнях з коротким промптом.

| Метод | Тренування, хв | Пік RSS, GB | Validation loss | Валідний JSON, % | Category, % | Intent, % |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| LoRA | 1.120 | 4.279 | 0.5500 | 93.75 | 25.00 | 18.75 |
| QLoRA | 0.903 | 4.251 | 0.7640 | 100.00 | 0.00 | 12.50 |

У цьому запуску QLoRA швидша приблизно на 13.0 с. Піки RSS близькі, хоча байтів параметрів у NF4 значно менше. Пік RSS залежить також від тимчасових тензорів і стану процесу. QLoRA дала валідний JSON на всіх 16 запитах, але не вгадала жодної `category`. LoRA вгадала 4 категорії та 3 наміри. Отже, формат і точність слід оцінювати окремо. За 16 звернень одна відповідь змінює метрику на 6,25 відсоткового пункту, тому таблиця характеризує компактний запуск. У T4-профілі розділ 9 порівнює 4B з виміряною піковою VRAM.